# Corpus-regression sweep analysis

In [1]:
import polars as pl

from src import get_repo_base
from src.experiments.corpus_regression.analysis import (
    CorpusRegressionAnalysisConfig,
    plot_methods_vs_epoch,
    plot_methods_vs_lookforward,
)
from src.experiments.corpus_regression.config import artifacts_dir

ARTIFACTS = artifacts_dir()

# Print summarize() tables in full (default polars truncation hides columns).
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(64)
pl.Config.set_fmt_str_lengths(80)

polars.config.Config

## Parameters

In [2]:
# === Parameters ===
NUM_SAMPLES: int = 100_000       # 100_000 or 500_000
GAUSSIAN_STDEV: float = 1.0      # 1.0 or 0.5
LABEL_TYPE: str = "token_id"   # "rademacher" or "token_id"
NORMALIZE_LABELS: bool = True   # True for [0,1]-normalized token_id labels
LABEL_RANGE: tuple[float, float] = (0.0, 1.0)  # target range when normalized
TRAIN_FROM_SCRATCH: bool = False  # True -> read from artifacts_dir()/<method>_scratch/

# Derived output path — namespaced by combo to avoid overwriting
_combo_slug = f"n{NUM_SAMPLES // 1000}k_sigma{GAUSSIAN_STDEV}"
if LABEL_TYPE != "rademacher":
    _combo_slug += f"_{LABEL_TYPE}"
if NORMALIZE_LABELS:
    _combo_slug += "_normalized"
if TRAIN_FROM_SCRATCH:
    _combo_slug += "_scratch"
WRITEUP_ASSETS = get_repo_base() / "writeup" / "assets" / "corpus-regression" / _combo_slug

## Supervised-learning

In [3]:
sl = CorpusRegressionAnalysisConfig.from_sl_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    train_from_scratch=TRAIN_FROM_SCRATCH,
)
if sl is None:
    print("SL: no artifacts")
else:
    sl.describe("SL")
    print(sl.summarize(metric="corr"))
    print(sl.summarize(metric="mse"))
    display(
        sl.plot_vs_epoch(
            "corr",
            title="SL: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_corr.html",
        )
    )
    display(
        sl.plot_vs_epoch(
            "mse",
            title="SL: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_mse.html",
        )
    )
    display(
        sl.plot_vs_lookforward(
            title="SL: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_vs_lookforward.html",
        )
    )

SL                                   3 runs     1 groups   up to 3 seeds/group
shape: (1, 5)
┌────────┬─────────────────┬────────────┬─────────┬───────────────┐
│ study  ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_corr_mean │
│ ---    ┆ ---             ┆ ---        ┆ ---     ┆ ---           │
│ str    ┆ i64             ┆ i64        ┆ i64     ┆ f64           │
╞════════╪═════════════════╪════════════╪═════════╪═══════════════╡
│ look=1 ┆ 1               ┆ 2999       ┆ 3       ┆ 0.507644      │
└────────┴─────────────────┴────────────┴─────────┴───────────────┘
shape: (1, 5)
┌────────┬─────────────────┬────────────┬─────────┬──────────────┐
│ study  ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_mse_mean │
│ ---    ┆ ---             ┆ ---        ┆ ---     ┆ ---          │
│ str    ┆ i64             ┆ i64        ┆ i64     ┆ f64          │
╞════════╪═════════════════╪════════════╪═════════╪══════════════╡
│ look=1 ┆ 1               ┆ 2999       ┆ 3       ┆ 0.026745     │
└────────┴─────

In [4]:
# SL: per-step training loss / MSE (dense per-gradient-step curves; requires step-level artifacts)
if sl is not None:
    try:
        display(sl.plot_vs_step("loss", title="SL: per-step loss", save_path=WRITEUP_ASSETS / "sl_per_step_loss.html"))
        display(sl.plot_vs_step("mse", title="SL: per-step MSE", save_path=WRITEUP_ASSETS / "sl_per_step_mse.html"))
    except ValueError as e:
        print(f"SL per-step plots unavailable (old-format artifacts?): {e}")

## SL+NTP-CE (token-level cross-entropy supervised)

Drop-in CE baseline for SL/MSE. Full-param fine-tunes the pretrained
`AutoModelForCausalLM` with cross-entropy on the lookahead token id (K=1
only). Validation reuses the projection from `ntp_baseline.py`:
`softmax(logits) @ label_projector` → MSE, so `sl_ce` lands in the same
target space as SL/GRPO/RLOO/MaxRL and is directly comparable.

In [5]:
sl_ce = CorpusRegressionAnalysisConfig.from_sl_ce_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    train_from_scratch=TRAIN_FROM_SCRATCH,
)
if sl_ce is None:
    print("SL+NTP-CE: no artifacts")
else:
    sl_ce.describe("SL+NTP-CE")
    print(sl_ce.summarize(metric="corr"))
    print(sl_ce.summarize(metric="mse"))
    display(
        sl_ce.plot_vs_epoch(
            "corr",
            title="SL+NTP-CE: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_ce_per_epoch_corr.html",
        )
    )
    display(
        sl_ce.plot_vs_epoch(
            "mse",
            title="SL+NTP-CE: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_ce_per_epoch_mse.html",
        )
    )
    display(
        sl_ce.plot_vs_lookforward(
            title="SL+NTP-CE: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_ce_vs_lookforward.html",
        )
    )

SL+NTP-CE: no artifacts


In [6]:
# SL+NTP-CE: per-step training loss / MSE (CE loss; projected MSE diagnostic)
if sl_ce is not None:
    try:
        display(sl_ce.plot_vs_step("loss", title="SL+NTP-CE: per-step loss (CE)", save_path=WRITEUP_ASSETS / "sl_ce_per_step_loss.html"))
        display(sl_ce.plot_vs_step("mse", title="SL+NTP-CE: per-step MSE (projected)", save_path=WRITEUP_ASSETS / "sl_ce_per_step_mse.html"))
    except ValueError as e:
        print(f"SL+NTP-CE per-step plots unavailable (old-format artifacts?): {e}")

## GRPO

In [7]:
grpo = CorpusRegressionAnalysisConfig.from_grpo_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE, normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    train_from_scratch=TRAIN_FROM_SCRATCH,
)
if grpo is None:
    print("GRPO: no artifacts")
else:
    grpo.describe("GRPO")
    print(grpo.summarize(metric="corr"))
    print(grpo.summarize(metric="mse"))
    display(
        grpo.plot_vs_epoch(
            "corr",
            title="GRPO: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_corr.html",
        )
    )
    display(
        grpo.plot_vs_epoch(
            "mse",
            title="GRPO: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_mse.html",
        )
    )
    display(
        grpo.plot_vs_lookforward(
            title="GRPO: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_vs_lookforward.html",
        )
    )

GRPO                                 3 runs     1 groups   up to 3 seeds/group
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬───────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_corr_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---           │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64           │
╞═══════════════╪═════════════════╪════════════╪═════════╪═══════════════╡
│ look=1 r=1024 ┆ 1               ┆ 5999       ┆ 3       ┆ 0.485545      │
└───────────────┴─────────────────┴────────────┴─────────┴───────────────┘
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬──────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_mse_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---          │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64          │
╞═══════════════╪═════════════════╪════════════╪═════════╪══════════════

In [8]:
# GRPO: per-step training loss / MSE
if grpo is not None:
    try:
        display(grpo.plot_vs_step("loss", title="GRPO: per-step loss", save_path=WRITEUP_ASSETS / "grpo_per_step_loss.html"))
        display(grpo.plot_vs_step("mse", title="GRPO: per-step MSE", save_path=WRITEUP_ASSETS / "grpo_per_step_mse.html"))
    except ValueError as e:
        print(f"GRPO per-step plots unavailable (old-format artifacts?): {e}")

## MaxRL (subtract-baseline + factorized)

In [9]:
maxrl_sf = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
    artifacts_root=ARTIFACTS,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
    train_from_scratch=TRAIN_FROM_SCRATCH,
)
if maxrl_sf is None:
    print("MaxRL (sub-baseline, factorized): no artifacts")
else:
    maxrl_sf.describe("MaxRL (sub-baseline, factorized)")
    print(maxrl_sf.summarize(metric="corr"))
    print(maxrl_sf.summarize(metric="mse"))
    display(
        maxrl_sf.plot_vs_epoch(
            "corr",
            title="MaxRL (sub-baseline, factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_corr.html",
        )
    )
    display(
        maxrl_sf.plot_vs_epoch(
            "mse",
            title="MaxRL (sub-baseline, factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_mse.html",
        )
    )
    display(
        maxrl_sf.plot_vs_lookforward(
            title="MaxRL (sub-baseline, factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_vs_lookforward.html",
        )
    )

MaxRL (sub-baseline, factorized)     3 runs     1 groups   up to 3 seeds/group
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬───────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_corr_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---           │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64           │
╞═══════════════╪═════════════════╪════════════╪═════════╪═══════════════╡
│ look=1 r=1024 ┆ 1               ┆ 1999       ┆ 3       ┆ 0.499582      │
└───────────────┴─────────────────┴────────────┴─────────┴───────────────┘
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬──────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_mse_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---          │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64          │
╞═══════════════╪═════════════════╪════════════╪═════════╪══════════════

In [10]:
# MaxRL: per-step training loss / MSE
if maxrl_sf is not None:
    try:
        display(maxrl_sf.plot_vs_step("loss", title="MaxRL (sub-baseline, factorized): per-step loss", save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_step_loss.html"))
        display(maxrl_sf.plot_vs_step("mse", title="MaxRL (sub-baseline, factorized): per-step MSE", save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_step_mse.html"))
    except ValueError as e:
        print(f"MaxRL per-step plots unavailable (old-format artifacts?): {e}")

### Other MaxRL ablations

Uncomment to inspect the other (`subtract_baseline`, `use_factorized_likelihoods`) combinations if we run these sweeps

In [11]:
# for sub, fact in [(True, False), (False, True), (False, False)]:
#     cfg = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
#         artifacts_root=ARTIFACTS,
#         subtract_baseline=sub,
#         use_factorized_likelihoods=fact,
#         num_samples=NUM_SAMPLES,
#         gaussian_stdev=GAUSSIAN_STDEV,
#     )
#     label = f"MaxRL (sub={sub}, fact={fact})"
#     if cfg is None:
#         print(f"{label}: no artifacts")
#         continue
#     cfg.describe(label)
#     display(cfg.plot_vs_epoch("corr", title=f"{label}: per-epoch (corr)", show_seed_bar=True))
#     display(cfg.plot_vs_lookforward(title=f"{label}: best-epoch vs num_lookforward_tokens", x_scale="uniform", show_seed_bar=True))

## RLOO (factorized)

In [12]:
rloo_f = CorpusRegressionAnalysisConfig.from_rloo_sweep(
    artifacts_root=ARTIFACTS,
    factorized=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
    train_from_scratch=TRAIN_FROM_SCRATCH,
)
if rloo_f is None:
    print("RLOO (factorized): no artifacts")
else:
    rloo_f.describe("RLOO (factorized)")
    print(rloo_f.summarize(metric="corr"))
    print(rloo_f.summarize(metric="mse"))
    display(
        rloo_f.plot_vs_epoch(
            "corr",
            title="RLOO (factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_corr.html",
        )
    )
    display(
        rloo_f.plot_vs_epoch(
            "mse",
            title="RLOO (factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_mse.html",
        )
    )
    display(
        rloo_f.plot_vs_lookforward(
            title="RLOO (factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_vs_lookforward.html",
        )
    )

RLOO (factorized)                    3 runs     1 groups   up to 3 seeds/group
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬───────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_corr_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---           │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64           │
╞═══════════════╪═════════════════╪════════════╪═════════╪═══════════════╡
│ look=1 r=1024 ┆ 1               ┆ 2999       ┆ 3       ┆ 0.497011      │
└───────────────┴─────────────────┴────────────┴─────────┴───────────────┘
shape: (1, 5)
┌───────────────┬─────────────────┬────────────┬─────────┬──────────────┐
│ study         ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ val_mse_mean │
│ ---           ┆ ---             ┆ ---        ┆ ---     ┆ ---          │
│ str           ┆ i64             ┆ i64        ┆ i64     ┆ f64          │
╞═══════════════╪═════════════════╪════════════╪═════════╪══════════════

In [13]:
# RLOO: per-step training loss / MSE
if rloo_f is not None:
    try:
        display(rloo_f.plot_vs_step("loss", title="RLOO (factorized): per-step loss", save_path=WRITEUP_ASSETS / "rloo_factorized_per_step_loss.html"))
        display(rloo_f.plot_vs_step("mse", title="RLOO (factorized): per-step MSE", save_path=WRITEUP_ASSETS / "rloo_factorized_per_step_mse.html"))
    except ValueError as e:
        print(f"RLOO per-step plots unavailable (old-format artifacts?): {e}")

## NTP baseline (intrinsic-variance proxy)

Inference-only: deterministic given the dataset (no seeds, single epoch).
We assemble one config across `candidate_lookforward_tokens` so it slots
into the same plotting helpers as the trained methods.

In [14]:
from src.data.corpus_regression import candidate_lookforward_tokens

_ntp_per_look = [
    CorpusRegressionAnalysisConfig.from_ntp_baseline(
        artifacts_root=ARTIFACTS,
        num_lookforward_tokens=n,
        num_samples=NUM_SAMPLES,
        label_type=LABEL_TYPE,
        normalize_labels=NORMALIZE_LABELS,
        label_range=LABEL_RANGE,
    )
    for n in candidate_lookforward_tokens
]
_ntp_per_look = [c for c in _ntp_per_look if c is not None]

if not _ntp_per_look:
    ntp = None
    print("NTP baseline: no artifacts")
else:
    ntp = CorpusRegressionAnalysisConfig(
        studies={k: v for c in _ntp_per_look for k, v in c.studies.items()},
        study_seeds={k: v for c in _ntp_per_look for k, v in c.study_seeds.items()},
        study_lookforwards={
            k: v for c in _ntp_per_look for k, v in c.study_lookforwards.items()
        },
    )
    ntp.describe("NTP baseline")
    print(ntp.summarize(metric="corr"))
    print(ntp.summarize(metric="mse"))
    display(
        ntp.plot_vs_lookforward(
            title="NTP baseline: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=False,
            save_path=WRITEUP_ASSETS / "ntp_baseline_vs_lookforward.html",
        )
    )
    display(
        ntp.plot_vs_lookforward(
            metric="mse",
            title="NTP baseline: best-epoch MSE vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=False,
            save_path=WRITEUP_ASSETS / "ntp_baseline_vs_lookforward_mse.html",
        )
    )

NTP baseline                         1 runs     1 groups   up to 1 seeds/group
shape: (1, 6)
┌────────┬─────────────────┬────────────┬─────────┬─────────────────┬───────────────┐
│ study  ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ train_corr_mean ┆ val_corr_mean │
│ ---    ┆ ---             ┆ ---        ┆ ---     ┆ ---             ┆ ---           │
│ str    ┆ i64             ┆ i64        ┆ i64     ┆ f64             ┆ f64           │
╞════════╪═════════════════╪════════════╪═════════╪═════════════════╪═══════════════╡
│ look=1 ┆ 1               ┆ 0          ┆ 1       ┆ 0.705886        ┆ 0.710171      │
└────────┴─────────────────┴────────────┴─────────┴─────────────────┴───────────────┘
shape: (1, 6)
┌────────┬─────────────────┬────────────┬─────────┬────────────────┬──────────────┐
│ study  ┆ num_lookforward ┆ best_epoch ┆ n_seeds ┆ train_mse_mean ┆ val_mse_mean │
│ ---    ┆ ---             ┆ ---        ┆ ---     ┆ ---            ┆ ---          │
│ str    ┆ i64             ┆ i64       

## Cross-method comparison

In [15]:
if any(c is not None for c in (sl, sl_ce, grpo, maxrl_sf, rloo_f, ntp)):
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            ntp_baseline=ntp,
            title="Methods: best-epoch corr vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward.html",
        )
    )
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            ntp_baseline=ntp,
            metric="mse",
            title="Methods: best-epoch MSE vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward_mse.html",
        )
    )
else:
    print("No artifacts for any method.")

### Per-epoch training curves (look=1, all methods)

In [16]:
if any(c is not None for c in (sl, sl_ce, grpo, maxrl_sf, rloo_f, ntp)):
    display(
        plot_methods_vs_epoch(
            sl=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            ntp_baseline=ntp,
            num_lookforward_tokens=1,
            metric="corr",
            show_seed_bar=True,
            title="Methods: per-epoch corr (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_corr.html",
        )
    )
    display(
        plot_methods_vs_epoch(
            sl=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            ntp_baseline=ntp,
            num_lookforward_tokens=1,
            metric="mse",
            show_seed_bar=True,
            title="Methods: per-epoch MSE (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_mse.html",
        )
    )
else:
    print("No artifacts for any method.")